# CV-STQA — M2: CNN + Temporal Transformer

**Independent of M1.** Uses the same train/validation/test split, videos, frame sampling, and six STQA targets.

**Pipeline:** Video → 16 frames → frozen ResNet-50 → 16×2048 features → 512-D projection → positional encoding → Transformer Encoder → temporal mean pooling → 6 quality scores.

In [1]:
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
import cv2
from PIL import Image

from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr, kendalltau

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
SEED = 42

NUM_FRAMES = 16
IMAGE_SIZE = 224

CNN_DIM = 2048
D_MODEL = 512
NHEAD = 8
NUM_LAYERS = 2
DIM_FEEDFORWARD = 1024
DROPOUT = 0.10

BATCH_SIZE = 4
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_ALIASES = {
    "TS": ["TS", "ts", "temporal_stability", "temporal_stability_score"],
    "V": ["V", "v", "visibility", "visibility_score"],
    "B": ["B", "b", "brightness", "brightness_score"],
    "S": ["S", "s", "sharpness", "sharpness_score"],
    "R": ["R", "r", "relevance", "relevance_score"],
    "O": ["O", "o", "overall", "overall_quality", "mos"]
}

VIDEO_ALIASES = [
    "video_id", "video", "Video", "video_name", "filename",
    "file_name", "id", "ID", "name"
]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

print("Device:", DEVICE)
print("Frames/video:", NUM_FRAMES)

Device: cuda
Frames/video: 16


## 1. Load the same Kaggle inputs used for M1

In [3]:
INPUT_ROOT = Path("/kaggle/input")

for p in sorted(INPUT_ROOT.iterdir()):
    print(p)

/kaggle/input/datasets


In [4]:
def find_one(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found under /kaggle/input. "
            "Make sure the corresponding Kaggle input is attached."
        )
    return matches[0]

TRAIN_CSV = find_one("train.csv")
VAL_CSV = find_one("val.csv")
TEST_CSV = find_one("test.csv")
M0_CONFIG = find_one("M0_config.json")

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

with open(M0_CONFIG, "r") as f:
    m0_config = json.load(f)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)
print("Columns:", train_df.columns.tolist())

Train: (700, 8)
Val:   (150, 8)
Test:  (150, 8)
Columns: ['video_id', 'TSV', 'B', 'SR', 'S', 'U', 'O', 'video_path']


In [7]:
# CELL — Identify video + STQA target columns

VIDEO_COL = "video_id"

TARGET_COLS = [
    "TSV",
    "B",
    "SR",
    "S",
    "U",
    "O"
]

# Verify required columns
required_cols = [VIDEO_COL] + TARGET_COLS

missing = [
    c for c in required_cols
    if c not in train_df.columns
]

if missing:
    print("Available columns:", train_df.columns.tolist())
    raise ValueError(f"Missing columns: {missing}")

print("Video column:", VIDEO_COL)
print("Target columns:", TARGET_COLS)

display(
    train_df[required_cols].head()
)

Video column: video_id
Target columns: ['TSV', 'B', 'SR', 'S', 'U', 'O']


,video_id,TSV,B,SR,S,U,O
0,68-11.mp4,2.0025,2.1235,1.8600,1.72120,1.8126,3.5140
1,48-3.mp4,2.1210,3.0000,1.6898,2.30670,2.3584,4.1275
2,2-10.mp4,1.4000,1.6566,1.3648,0.68874,1.4057,2.9560
3,40-1.mp4,1.8593,0.8656,0.4560,1.27030,1.0326,2.2650
4,52-15.mp4,1.6436,1.7560,0.8960,1.98970,1.6381,3.3420


## 2. Locate MAC-VQA videos and verify the split

In [8]:
VIDEO_EXTENSIONS = {".mp4",".avi",".mov",".mkv",".webm",".MP4",".AVI",".MOV",".MKV",".WEBM"}

candidate_dirs = []
for p in INPUT_ROOT.rglob("*"):
    if p.is_dir():
        try:
            if any(x.is_file() and x.suffix in VIDEO_EXTENSIONS for x in p.iterdir()):
                candidate_dirs.append(p)
        except:
            pass

if not candidate_dirs:
    raise FileNotFoundError("No directory containing video files was found.")

VIDEO_ROOT = max(
    candidate_dirs,
    key=lambda d: sum(1 for x in d.rglob("*") if x.is_file() and x.suffix in VIDEO_EXTENSIONS)
)

video_files = [
    x for x in VIDEO_ROOT.rglob("*")
    if x.is_file() and x.suffix in VIDEO_EXTENSIONS
]

video_lookup = {}
for p in video_files:
    video_lookup[p.name] = p
    video_lookup[p.stem] = p

print("Video root:", VIDEO_ROOT)
print("Videos found:", len(video_files))

Video root: /kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos
Videos found: 1000


In [9]:
def resolve_video_path(video_id):
    vid = str(video_id).strip()

    if vid in video_lookup:
        return video_lookup[vid]

    for ext in VIDEO_EXTENSIONS:
        p = VIDEO_ROOT / f"{vid}{ext}"
        if p.exists():
            return p

    for ext in VIDEO_EXTENSIONS:
        matches = list(VIDEO_ROOT.rglob(f"{vid}{ext}"))
        if matches:
            return matches[0]

    return None

def verify(df, name):
    sample = df[VIDEO_COL].head(min(20, len(df)))
    found = sum(resolve_video_path(v) is not None for v in sample)
    print(f"{name}: {found}/{len(sample)} sample videos found")
    return found == len(sample)

if not (verify(train_df,"Train") and verify(val_df,"Val") and verify(test_df,"Test")):
    raise RuntimeError("Video IDs could not be resolved. Stop before training.")

Train: 20/20 sample videos found
Val: 20/20 sample videos found
Test: 20/20 sample videos found


## 3. Extract and cache frozen ResNet-50 frame features

In [10]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=weights)
resnet = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE)
resnet.eval()

for p in resnet.parameters():
    p.requires_grad = False

print("Frozen ResNet-50 ready.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 228MB/s]


Frozen ResNet-50 ready.


In [11]:
def sample_video_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read {video_path}")

    indices = np.linspace(0, total - 1, num_frames).astype(int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))

    cap.release()

    if not frames:
        raise RuntimeError(f"No frames read from {video_path}")

    while len(frames) < num_frames:
        frames.append(frames[-1].copy())

    return frames[:num_frames]

@torch.inference_mode()
def extract_features(video_path):
    frames = sample_video_frames(video_path)
    batch = torch.stack([transform(f) for f in frames]).to(DEVICE)
    return resnet(batch).flatten(1).cpu().float()

sample_path = resolve_video_path(train_df.iloc[0][VIDEO_COL])
sample_feature = extract_features(sample_path)

print("Feature shape:", tuple(sample_feature.shape))
assert tuple(sample_feature.shape) == (16, 2048)

Feature shape: (16, 2048)


In [12]:
CACHE_DIR = Path("/kaggle/working/M2_feature_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def cache_path(video_id):
    safe = str(video_id).replace("/","_").replace("\\","_").replace(":","_")
    return CACHE_DIR / f"{safe}.pt"

def build_cache(df, split):
    created = skipped = 0
    errors = []

    for video_id in tqdm(df[VIDEO_COL], desc=f"Caching {split}"):
        out = cache_path(video_id)

        if out.exists():
            skipped += 1
            continue

        path = resolve_video_path(video_id)
        if path is None:
            errors.append(str(video_id))
            continue

        try:
            torch.save(extract_features(path), out)
            created += 1
        except Exception as e:
            errors.append(f"{video_id}: {e}")

    print(f"{split}: created={created}, cached={skipped}, errors={len(errors)}")
    if errors:
        print("First errors:", errors[:5])
    return errors

errors = (
    build_cache(train_df,"train") +
    build_cache(val_df,"val") +
    build_cache(test_df,"test")
)

if errors:
    raise RuntimeError("Feature extraction had errors. Fix them before training.")

Caching train:   0%|          | 0/700 [00:00<?, ?it/s]

train: created=700, cached=0, errors=0


Caching val:   0%|          | 0/150 [00:00<?, ?it/s]

val: created=150, cached=0, errors=0


Caching test:   0%|          | 0/150 [00:00<?, ?it/s]

test: created=150, cached=0, errors=0


## 4. Build the M2 Transformer

In [13]:
class FeatureDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = str(row[VIDEO_COL])

        features = torch.load(
            cache_path(video_id),
            map_location="cpu",
            weights_only=True
        ).float()

        targets = torch.tensor(
            row[TARGET_COLS].astype(float).values,
            dtype=torch.float32
        )

        return features, targets, video_id

train_ds = FeatureDataset(train_df)
val_ds = FeatureDataset(val_df)
test_ds = FeatureDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=(NUM_WORKERS > 0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         persistent_workers=(NUM_WORKERS > 0))

x, y, vid = train_ds[0]
print("Video:", vid)
print("Features:", tuple(x.shape))
print("Targets:", tuple(y.shape))

Video: 68-11.mp4
Features: (16, 2048)
Targets: (6,)


In [14]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=NUM_FRAMES, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class M2TemporalTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(CNN_DIM, D_MODEL),
            nn.LayerNorm(D_MODEL),
            nn.GELU()
        )

        self.positional = PositionalEncoding(D_MODEL)

        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NHEAD,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=NUM_LAYERS)

        self.head = nn.Sequential(
            nn.LayerNorm(D_MODEL),
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 6)
        )

    def forward(self, x):
        x = self.projection(x)
        x = self.positional(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)

model = M2TemporalTransformer().to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Total parameters: 5,389,574
Trainable parameters: 5,389,574


In [15]:
# Forward-pass sanity check
bx, by, _ = next(iter(train_loader))
bx = bx.to(DEVICE)

with torch.no_grad():
    bp = model(bx)

print("Input :", tuple(bx.shape))
print("Output:", tuple(bp.shape))
assert bp.shape[-1] == 6
print("PASSED")

Input : (4, 16, 2048)
Output: (4, 6)
PASSED


## 5. Train M2

In [16]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

def train_epoch():
    model.train()
    total_loss = 0.0

    for features, targets, _ in tqdm(train_loader, desc="Train", leave=False):
        features = features.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(features)
        loss = criterion(pred, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

@torch.no_grad()
def val_epoch():
    model.eval()
    total_loss = 0.0

    for features, targets, _ in val_loader:
        features = features.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        total_loss += criterion(model(features), targets).item()

    return total_loss / len(val_loader)

In [17]:
MODEL_DIR = Path("/kaggle/working/M2")
MODEL_DIR.mkdir(exist_ok=True)

BEST_PATH = MODEL_DIR / "m2_best.pt"
history = []
best_val = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch()
    val_loss = val_epoch()
    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]
    history.append([epoch, train_loss, val_loss, lr])

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train {train_loss:.6f} | "
        f"Val {val_loss:.6f} | LR {lr:.2e}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "target_cols": TARGET_COLS
        }, BEST_PATH)
        print("  -> best model saved")

history_df = pd.DataFrame(
    history, columns=["epoch","train_loss","val_loss","learning_rate"]
)
history_df.to_csv(MODEL_DIR / "m2_training_history.csv", index=False)

print("Best validation loss:", best_val)

Train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 01/10 | Train 0.362136 | Val 0.249065 | LR 1.00e-04
  -> best model saved


Train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 02/10 | Train 0.182961 | Val 0.162739 | LR 1.00e-04
  -> best model saved


Train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 03/10 | Train 0.158076 | Val 0.271975 | LR 1.00e-04


... (10 outputs left out)

Train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 09/10 | Train 0.084903 | Val 0.142498 | LR 1.00e-04


Train:   0%|          | 0/175 [00:00<?, ?it/s]

Epoch 10/10 | Train 0.082970 | Val 0.141124 | LR 5.00e-05
Best validation loss: 0.13788912543340734


## 6. Test and save M2 results

In [18]:
checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

@torch.no_grad()
def predict():
    model.eval()
    preds, trues, ids = [], [], []

    for features, targets, video_ids in tqdm(test_loader, desc="Test"):
        features = features.to(DEVICE, non_blocking=True)
        preds.append(model(features).cpu().numpy())
        trues.append(targets.numpy())
        ids.extend(video_ids)

    return np.concatenate(preds), np.concatenate(trues), ids

predictions, targets, video_ids = predict()

print("Predictions:", predictions.shape)
print("Targets:", targets.shape)

Test:   0%|          | 0/38 [00:00<?, ?it/s]

Predictions: (150, 6)
Targets: (150, 6)


In [19]:
def corr(fn, a, b):
    try:
        v = fn(a,b)[0]
        return float(v) if np.isfinite(v) else np.nan
    except:
        return np.nan

rows = []
dimensions = ["TS","V","B","S","R","O"]

for i, dim in enumerate(dimensions):
    yt = targets[:,i]
    yp = predictions[:,i]

    rows.append({
        "Dimension": dim,
        "PLCC": corr(pearsonr, yt, yp),
        "SRCC": corr(spearmanr, yt, yp),
        "KRCC": corr(kendalltau, yt, yp),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp)))
    })

metrics_df = pd.DataFrame(rows)

metrics_df.loc[len(metrics_df)] = [
    "Overall Mean",
    metrics_df["PLCC"].mean(),
    metrics_df["SRCC"].mean(),
    metrics_df["KRCC"].mean(),
    metrics_df["RMSE"].mean()
]

display(metrics_df)

,Dimension,PLCC,SRCC,KRCC,RMSE
0,TS,0.902855,0.893705,0.727469,0.305073
1,V,0.768045,0.732535,0.556016,0.523458
2,B,0.833067,0.827107,0.637397,0.378532
3,S,0.911673,0.909457,0.735839,0.321700
4,R,0.892670,0.897206,0.715124,0.329225
5,O,0.936172,0.943316,0.794215,0.371760
6,Overall Mean,0.874080,0.867221,0.694343,0.371625


In [20]:
pred_df = pd.DataFrame({"video_id": video_ids})

for i, dim in enumerate(dimensions):
    pred_df[f"true_{dim}"] = targets[:,i]
    pred_df[f"pred_{dim}"] = predictions[:,i]

metrics_df.to_csv(MODEL_DIR / "m2_test_metrics.csv", index=False)
pred_df.to_csv(MODEL_DIR / "m2_test_predictions.csv", index=False)

config = {
    "model": "M2_CNN_Temporal_Transformer",
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "cnn": "ResNet-50 frozen ImageNet",
    "cnn_feature_dim": CNN_DIM,
    "d_model": D_MODEL,
    "nhead": NHEAD,
    "num_layers": NUM_LAYERS,
    "feedforward_dim": DIM_FEEDFORWARD,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "targets": dimensions,
    "best_epoch": int(checkpoint["epoch"]),
    "best_val_loss": float(checkpoint["val_loss"])
}

with open(MODEL_DIR / "m2_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("M2 files saved:")
for p in sorted(MODEL_DIR.iterdir()):
    print(" -", p)

M2 files saved:
 - /kaggle/working/M2/m2_best.pt
 - /kaggle/working/M2/m2_config.json
 - /kaggle/working/M2/m2_test_metrics.csv
 - /kaggle/working/M2/m2_test_predictions.csv
 - /kaggle/working/M2/m2_training_history.csv


## What to save from M2

After the notebook finishes, keep these files:

- `m2_best.pt` — trained M2 Transformer
- `m2_test_metrics.csv` — PLCC/SRCC/KRCC/RMSE
- `m2_test_predictions.csv` — test predictions
- `m2_training_history.csv` — training/validation losses
- `m2_config.json` — exact M2 configuration

**Do not use M1 predictions as M2 input.** M1 and M2 are independent experiments.